# sqlite3 Basics

## Overview
Python's built-in `sqlite3` module provides a DB-API 2.0 compliant interface to SQLite. It is the foundation for all SQL interaction in Python and shares the same interface pattern as `psycopg2`, `cx_Oracle`, and other database drivers.

**DB-API 2.0 pattern (the same across all Python database drivers):**
```
connect()  →  connection  →  cursor  →  execute()  →  fetch  →  close()
```

**Connection vs Cursor:**
- **Connection** — represents the database session; manages transactions (`commit`, `rollback`)
- **Cursor** — executes SQL statements; holds the result set; created from a connection

**Mixed dataset:** patients (healthcare) and transactions (finance).

---

In [1]:
import sqlite3

conn = sqlite3.connect(":memory:")
conn.execute("PRAGMA journal_mode=WAL")
conn.execute("PRAGMA foreign_keys = ON")
conn.row_factory = sqlite3.Row              # rows behave like dicts

conn.executescript("""
CREATE TABLE patients (
    patient_id   TEXT PRIMARY KEY,
    first_name   TEXT NOT NULL,
    last_name    TEXT NOT NULL,
    province     TEXT NOT NULL,
    dob          TEXT NOT NULL,
    sex          TEXT NOT NULL CHECK(sex IN ('M','F','Other'))
);
CREATE TABLE transactions (
    tx_id        INTEGER PRIMARY KEY AUTOINCREMENT,
    account_id   TEXT NOT NULL,
    patient_id   TEXT,
    amount       REAL NOT NULL,
    tx_type      TEXT NOT NULL,
    tx_date      TEXT NOT NULL DEFAULT (date('now')),
    description  TEXT
);
INSERT INTO patients VALUES
    ('P001','Aroha',  'Ngata',   'NB','1985-03-15','F'),
    ('P002','Liam',   'Chen',    'ON','1990-07-22','M'),
    ('P003','Fatima', 'Rashid',  'BC','1978-11-05','F'),
    ('P004','James',  'MacLeod', 'NB','2001-01-30','M'),
    ('P005','Maria',  'Santos',  'QC','1995-06-18','F');
INSERT INTO transactions(account_id,patient_id,amount,tx_type,description) VALUES
    ('ACC001','P001', 250.00,'payment','copay'),
    ('ACC001','P001',1200.00,'payment','specialist visit'),
    ('ACC002','P002',  80.00,'payment','copay'),
    ('ACC003','P003', 450.00,'payment','lab work'),
    ('ACC001','P001', 500.00,'refund', 'duplicate charge');
""")
conn.commit()
print("Schema ready")
print("sqlite3 version:", sqlite3.sqlite_version)
total_p = conn.execute("SELECT COUNT(*) FROM patients").fetchone()[0]
total_t = conn.execute("SELECT COUNT(*) FROM transactions").fetchone()[0]
print(f"  {total_p} patients, {total_t} transactions")


Schema ready
sqlite3 version: 3.45.3
  5 patients, 5 transactions


---
## Parameterised queries and row_factory

In [2]:
import sqlite3

conn = sqlite3.connect(":memory:")
conn.row_factory = sqlite3.Row
conn.executescript("""
CREATE TABLE patients (patient_id TEXT PRIMARY KEY, first_name TEXT NOT NULL,
    last_name TEXT NOT NULL, province TEXT NOT NULL, dob TEXT NOT NULL, sex TEXT NOT NULL);
INSERT INTO patients VALUES
    ('P001','Aroha','Ngata','NB','1985-03-15','F'),
    ('P002','Liam','Chen','ON','1990-07-22','M'),
    ('P003','Fatima','Rashid','BC','1978-11-05','F'),
    ('P004','James','MacLeod','NB','2001-01-30','M');
""")

# ── fetchall, fetchone, fetchmany ────────────────────────────────
print("NB patients (positional parameter):")
rows = conn.execute("SELECT * FROM patients WHERE province = ?", ("NB",)).fetchall()
for r in rows:
    print(f"  {r['patient_id']}  {r['first_name']} {r['last_name']}")

print()
print("fetchone (named parameter):")
row = conn.execute(
    "SELECT * FROM patients WHERE province = :prov AND sex = :sex",
    {"prov": "NB", "sex": "M"}
).fetchone()
print(f"  {dict(row) if row else 'No result'}")

print()
print("fetchmany(2):")
cur = conn.execute("SELECT patient_id, first_name FROM patients ORDER BY first_name")
batch = cur.fetchmany(2)
print(f"  {[dict(r) for r in batch]}")

# ── executemany for batch inserts ────────────────────────────────
print()
new_patients = [
    ("P005", "Maria",  "Santos", "QC", "1995-06-18", "F"),
    ("P006", "David",  "Park",   "AB", "1982-09-12", "M"),
]
conn.executemany("INSERT INTO patients VALUES (?,?,?,?,?,?)", new_patients)
conn.commit()
total = conn.execute("SELECT COUNT(*) FROM patients").fetchone()[0]
print(f"After executemany: {total} patients")


NB patients (positional parameter):
  P001  Aroha Ngata
  P004  James MacLeod

fetchone (named parameter):
  {'patient_id': 'P004', 'first_name': 'James', 'last_name': 'MacLeod', 'province': 'NB', 'dob': '2001-01-30', 'sex': 'M'}

fetchmany(2):
  [{'patient_id': 'P001', 'first_name': 'Aroha'}, {'patient_id': 'P003', 'first_name': 'Fatima'}]

After executemany: 6 patients


---
## Context managers and error handling

In [3]:
import sqlite3

conn = sqlite3.connect(":memory:")
conn.execute("PRAGMA foreign_keys = ON")
conn.executescript("""
CREATE TABLE patients (patient_id TEXT PRIMARY KEY, first_name TEXT,
    last_name TEXT, province TEXT, dob TEXT, sex TEXT);
INSERT INTO patients VALUES ('P001','Aroha','Ngata','NB','1985-03-15','F');
""")
conn.commit()

# ── 'with conn:' auto-commits on success, auto-rolls back on exception ──
print("=== Context manager: with conn: ===")
try:
    with conn:
        conn.execute("INSERT INTO patients VALUES (?,?,?,?,?,?)",
                     ("P002","Liam","Chen","ON","1990-07-22","M"))
        conn.execute("INSERT INTO patients VALUES (?,?,?,?,?,?)",
                     ("P002","Duplicate","Key","ON","1990-07-22","M"))  # UNIQUE violation
except sqlite3.IntegrityError as e:
    print(f"  Rolled back — IntegrityError: {e}")

count = conn.execute("SELECT COUNT(*) FROM patients").fetchone()[0]
print(f"  Patients after rollback: {count}  (P002 not inserted)")

with conn:
    conn.execute("INSERT INTO patients VALUES (?,?,?,?,?,?)",
                 ("P002","Liam","Chen","ON","1990-07-22","M"))
count = conn.execute("SELECT COUNT(*) FROM patients").fetchone()[0]
print(f"  Patients after correct insert: {count}")

print()
print("=== Useful connection settings ===")
settings = [
    ("conn.row_factory = sqlite3.Row",           "Rows accessible by column name"),
    ("conn.isolation_level = None",              "Autocommit mode (no implicit BEGIN)"),
    ("conn.execute('PRAGMA foreign_keys=ON')",   "Enable FK constraint enforcement"),
    ("conn.execute('PRAGMA journal_mode=WAL')",  "WAL mode: better concurrent reads"),
    ("conn.execute('PRAGMA cache_size=-64000')", "64 MB page cache (negative = KB)"),
]
for setting, desc in settings:
    print(f"  {setting:<48s}  {desc}")
conn.close()


=== Context manager: with conn: ===
  Rolled back — IntegrityError: UNIQUE constraint failed: patients.patient_id
  Patients after rollback: 1  (P002 not inserted)
  Patients after correct insert: 2

=== Useful connection settings ===
  conn.row_factory = sqlite3.Row                    Rows accessible by column name
  conn.isolation_level = None                       Autocommit mode (no implicit BEGIN)
  conn.execute('PRAGMA foreign_keys=ON')            Enable FK constraint enforcement
  conn.execute('PRAGMA journal_mode=WAL')           WAL mode: better concurrent reads
  conn.execute('PRAGMA cache_size=-64000')          64 MB page cache (negative = KB)


---
## Common Pitfalls

**1. String-formatting SQL instead of using parameterised queries**
Concatenating user input directly into a SQL string is a SQL injection vulnerability. Always use placeholder syntax: `?` for sqlite3, `:name` for SQLAlchemy `text()`. The driver handles quoting and escaping correctly; string formatting never does.

**2. Forgetting the trailing comma in single-element parameter tuples**
`conn.execute("... WHERE id = ?", (patient_id))` passes a string, not a tuple. Python sees `('P001')` as just `'P001'`. Write `(patient_id,)` -- the comma is required.

**3. Not committing after INSERT/UPDATE/DELETE**
sqlite3 implicitly begins a transaction on the first DML statement. Without `conn.commit()` or `with conn:`, changes are invisible to other connections and are lost when the connection closes.

**4. Leaving connections open indefinitely**
An unclosed sqlite3 connection holds a file lock. In scripts that open connections in loops without closing them, file descriptors are exhausted. Always call `conn.close()` or use `with sqlite3.connect(...) as conn:`.

**5. Using fetchall() on very large result sets**
`fetchall()` loads all rows into memory at once. For large queries use `fetchmany(N)` in a loop or iterate the cursor directly (`for row in cursor:`) to process rows in batches.

**6. row_factory set after queries have already run**
`conn.row_factory = sqlite3.Row` must be set before executing queries. Set it immediately after `connect()` so all cursors from that connection inherit the row factory.


---
*sql_methods_library - Samantha McGarrigle*